# RL basics

Термины и понятия:

- агент/среда
- наблюдение $o$ / состояние $s$
- действие $a$, стратегия $\pi: \pi(s) \rightarrow a$ функция перехода $T: T(s, a) \rightarrow s'$
- вознаграждение $r$, ф-я вознаграждений $R: R(s, a) \rightarrow r$
- цикл взаимодействия, траектория $\tau: (s_0, a_0, r_0, s_1, a_1, r_1, ..., s_T, a_T, r_T)$, эпизод
- отдача $G$, подсчет отдачи, средняя[/ожидаемая] отдача $\mathbb{E}[G]$

In [57]:
try:
    import google.colab
    COLAB = True
except ModuleNotFoundError:
    COLAB = False
    pass

if COLAB:
    !pip -q install "gymnasium[classic-control, atari, accept-rom-license]"
    !pip -q install piglet
    !pip -q install imageio_ffmpeg
    !pip -q install moviepy==1.0.3

In [58]:
!pip install gymnasium


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [59]:
import glob
import io
import base64
import gymnasium as gym
import numpy as np
from IPython import display as ipythondisplay
from IPython.display import HTML
import matplotlib.pyplot as plt
%matplotlib inline

## Agent, environment

<img src=https://gymnasium.farama.org/_images/lunar_lander.gif caption="lunar lander" width="150" height="50"><img src=https://gymnasium.farama.org/_images/mountain_car.gif caption="mountain car" width="150" height="50">
<img src=https://gymnasium.farama.org/_images/cliff_walking.gif caption="cliff walking" width="300" height="50">
<img src=https://ale.farama.org/_images/montezuma_revenge.gif caption="montezuma revenge" width="150" height="100">
<img src=https://github.com/danijar/crafter/raw/main/media/video.gif caption="crafter" width="150" height="100">
<img src=https://camo.githubusercontent.com/6df2ca438d8fe8aa7a132b859315147818c54af608f8609320c3c20e938acf48/68747470733a2f2f6d656469612e67697068792e636f6d2f6d656469612f344e78376759694d394e44724d724d616f372f67697068792e676966 caption="malmo minecraft" width="150" height="100">
<img src=https://images.ctfassets.net/kftzwdyauwt9/e0c0947f-1a44-4528-4a41450a9f0a/2d0e85871d58d02dbe01b2469d693d4a/table-03.gif caption="roboschool" width="150" height="100">
<img src=https://raw.githubusercontent.com/Tviskaron/mipt/master/2019/RL/02/mdp.png caption="Марковский процесс принятия решений" width="150" height="100">
<img src=https://minigrid.farama.org/_images/DoorKeyEnv.gif caption="minigrid" width="120" height="120">

## Observation, state

TODO:
- добавить примеры наблюдений/состояний (числа, векторы, картинки)
- интуитивное объяснение различия, положить пока, что наблюдение = состояние
- пространство состояний


В каждый момент времени среда имеет некоторое внутреннее состояние. Здесь слово "состояние" я употребил скорее в интуитивном понимании, чтобы обозначить, что среда изменчива (иначе какой смысл с ней взаимодействовать, если ничего не меняется). В обучении с подкреплением под термином состояние $s$ (или $s_t$, где $t$ — текущее время) подразумевают либо абстрактно информацию о "состоянии" среды, либо ее явное представление в виде данных, достаточные для полного описания "состояния". *NB: Здесь можно провести аналогию с компьютерными играми — файл сохранения игры как раз содержит информацию о "состоянии" мира игры, чтобы в будущем можно было продолжить с текущей точки, так что данные этого файла в целом можно с некоторой натяжкой считать состоянием (с натяжкой, потому что редко когда в сложных играх файлы сохранения содержат прямо вот всю информацию, так что после перезагрузки вы получите не совсем точную копию). При этом обычно подразумевается, что состояние не содержит в себе ничего лишнего, то есть это **минимальный** набор информации.*

Наблюдением $o$ называют то, что агент "видит" о текущем состоянии среды. Это не обязательно зрение, а вообще вся доступная ему информация (условно, со всех его органов чувств).

В общем случае наблюдение: кортеж/словарь многомерных векторов чисел.

In [60]:
print(gym.make("CartPole-v0").reset()[0].shape)
print(gym.make("MountainCar-v0").reset()[0].shape)

(4,)
(2,)


## Action, policy, transition function

Рассмотрим следующие MDP:

- A: <img src=https://i.ibb.co/mrCMVZLQ/mdp-a.png caption="A" width="400" height="100">
- B: <img src=https://i.ibb.co/GQ2tVtjC/mdp-b.png caption="B" width="400" height="100">

Links to all:
[A](https://i.ibb.co/mrCMVZLQ/mdp-a.png)
[B](https://i.ibb.co/GQ2tVtjC/mdp-b.png)
[C](https://i.ibb.co/Jj9LYHjP/mdp-c.png)


Давайте явно запишем пространства состояний $S$ и действий $A$, а также функцию перехода $T$ среды.

In [61]:
states = set(range(3))
actions = set(range(1))

print(f'{states=} | {actions=}')

T = {
    (0, 0): 1,
    (1, 0): 2,
    (2, 0): 2
}
print(f'Transition function {T=}')

A_mdp = states, actions, T

states={0, 1, 2} | actions={0}
Transition function T={(0, 0): 1, (1, 0): 2, (2, 0): 2}


Попробуйте записать функцию перехода в матричном виде:

In [62]:
import numpy as np

# функция для перевода
def transition_function_to_matrix(T):
    n_states = max(max(s, s_next) for (s, _), s_next in T.items()) + 1

    counts = np.zeros((n_states, n_states), dtype=float)
    actions_per_state = [0] * n_states

    for (s, a), s_next in T.items():
        counts[s, s_next] += 1
        actions_per_state[s] += 1

    P = np.zeros_like(counts)
    for s in range(n_states):
        if actions_per_state[s] > 0:
            P[s] = counts[s] / actions_per_state[s]
        else:
            P[s, s] = 1.0
    return P

P = transition_function_to_matrix(T)

print(f'Matrix Transition function\n {P}')

Matrix Transition function
 [[0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 1.]]


Как получить вероятность нахождения агента в состоянии (1) через N шагов? Что происходит с вероятностями нахождения в состояниях при $N \rightarrow \infty$

In [63]:
P0 = [1, 0, 0]

def P_a_n(state, steps, matrix, start):
    ans = np.dot(start, np.linalg.matrix_power(matrix, steps))
    print(f'Вероятность нахождения агента в состоянии {state} через {steps} шагов: {ans}')
    return

for n in [1, 10, 100, 1000, 10000, 100000]:
    P_a_n(1, n, P, P0)

Вероятность нахождения агента в состоянии 1 через 1 шагов: [0. 1. 0.]
Вероятность нахождения агента в состоянии 1 через 10 шагов: [0. 0. 1.]
Вероятность нахождения агента в состоянии 1 через 100 шагов: [0. 0. 1.]
Вероятность нахождения агента в состоянии 1 через 1000 шагов: [0. 0. 1.]
Вероятность нахождения агента в состоянии 1 через 10000 шагов: [0. 0. 1.]
Вероятность нахождения агента в состоянии 1 через 100000 шагов: [0. 0. 1.]


Задайте еще несколько MDP:

- C: <img height="100" src="https://i.ibb.co/Jj9LYHjP/mdp-c.png" width="400"/>

In [64]:
# Для B:

T_B = {
    (0, 0): 1,
    (0, 1): 2,
    (0, 2): 3,
    (1, 0): 1,
    (1, 1): 1,
    (1, 2): 1,
    (2, 0): 2,
    (2, 1): 2,
    (2, 2): 2,
    (3, 0): 3,
    (3, 1): 3,
    (3, 2): 3
}

states = set(range(4))
actions = set(range(3))

B_mdp = states, actions, T_B

P_B = transition_function_to_matrix(T_B)

print(f'Matrix Transition function\n {P_B}\n\n')

P_0_B = [1, 0, 0, 0]

for n in [1, 10, 100, 1000, 10000, 100000]:
    P_a_n(1, n, P_B, P_0_B)

Matrix Transition function
 [[0.         0.33333333 0.33333333 0.33333333]
 [0.         1.         0.         0.        ]
 [0.         0.         1.         0.        ]
 [0.         0.         0.         1.        ]]


Вероятность нахождения агента в состоянии 1 через 1 шагов: [0.         0.33333333 0.33333333 0.33333333]
Вероятность нахождения агента в состоянии 1 через 10 шагов: [0.         0.33333333 0.33333333 0.33333333]
Вероятность нахождения агента в состоянии 1 через 100 шагов: [0.         0.33333333 0.33333333 0.33333333]
Вероятность нахождения агента в состоянии 1 через 1000 шагов: [0.         0.33333333 0.33333333 0.33333333]
Вероятность нахождения агента в состоянии 1 через 10000 шагов: [0.         0.33333333 0.33333333 0.33333333]
Вероятность нахождения агента в состоянии 1 через 100000 шагов: [0.         0.33333333 0.33333333 0.33333333]


In [65]:
# Для C:

T_C = {
    (0, 0): 1,
    (0, 1): 2,
    (1, 0): 1,
    (1, 1): 3,
    (2, 0): 3,
    (2, 1): 2,
    (3, 0): 3,
    (3, 1): 3
}
P_C = transition_function_to_matrix(T_C)

states = set(range(4))
actions = set(range(2))

C_mdp = states, actions, T_C

print(f'Matrix Transition function\n {P_C}\n\n')

P_0_C = [1, 0, 0, 0]

for n in [1, 10, 100, 1000, 10000, 100000]:
    P_a_n(1, n, P_B, P_0_B)

Matrix Transition function
 [[0.  0.5 0.5 0. ]
 [0.  0.5 0.  0.5]
 [0.  0.  0.5 0.5]
 [0.  0.  0.  1. ]]


Вероятность нахождения агента в состоянии 1 через 1 шагов: [0.         0.33333333 0.33333333 0.33333333]
Вероятность нахождения агента в состоянии 1 через 10 шагов: [0.         0.33333333 0.33333333 0.33333333]
Вероятность нахождения агента в состоянии 1 через 100 шагов: [0.         0.33333333 0.33333333 0.33333333]
Вероятность нахождения агента в состоянии 1 через 1000 шагов: [0.         0.33333333 0.33333333 0.33333333]
Вероятность нахождения агента в состоянии 1 через 10000 шагов: [0.         0.33333333 0.33333333 0.33333333]
Вероятность нахождения агента в состоянии 1 через 100000 шагов: [0.         0.33333333 0.33333333 0.33333333]


Давайте попробуем задать двух агентов: случайного и оптимального (для каждой среды свой).

In [66]:
class Agent:
    def __init__(self, actions):
        self.rng = np.random.default_rng()
        self.actions = np.array(list(actions))

    def act(self, state):
        return self.rng.integers(len(self.actions))

В качестве дополнения, запишите стратегию агента

In [67]:
# Для A (оптимальный агент) (У нас всего одно действие – его и будет оптимально брать)
class Optimal_Agent_A:
    def __init__(self, actions):
        self.actions = np.array(list(actions))

    def act(self, state):
        return 0  
    
    
# Для B (оптимальный агент) (Любой переход будет в терминальную вершину)
class Optimal_Agent_B:
    def __init__(self, actions):
        self.actions = np.array(list(actions))

    def act(self, state):
        return self.rng.integers(len(self.actions))
    
# Для C (оптимлальный агент) (Можно идти по действию равному четности чтобы долго не ифать)
class Optimal_Agent_C:
    def __init__(self, actions):
        self.actions = np.array(list(actions))

    def act(self, state):
        return int(state % 2)


## Reward, reward function

Теперь добавим произвольную функцию вознаграждения. Например, для A:

In [68]:
R = {
    (0, 0): -0.1,
    (1, 0): 1.0,
    (2, 0): 0.0
}
print(R)

A_mdp = *A_mdp, R
print(A_mdp)

R_B = {
    (0, 0): 0.1,
    (0, 1): 0.1,
    (0, 2): 0.1,
    (1, 0): 0.0,
    (1, 1): 0.0,
    (1, 2): 0.0,
    (2, 0): 0.0,
    (2, 1): 0.0,
    (2, 2): 0.0,
    (3, 0): 0.0,
    (3, 1): 0.0,
    (3, 2): 0.0
}

print(R_B)
B_mdp = *B_mdp, R_B
print(B_mdp)

R_C = {
    (0, 0): 0.1,
    (0, 1): 0.1,
    (1, 0): -0.1,
    (1, 1): 0.1,
    (2, 0): 0.1,
    (2, 1): -0.1,
    (3, 0): 0.0,
    (3, 1): 0.0
}

print(R_C)
C_mdp = *C_mdp, R_C
print(C_mdp)

{(0, 0): -0.1, (1, 0): 1.0, (2, 0): 0.0}
({0, 1, 2}, {0}, {(0, 0): 1, (1, 0): 2, (2, 0): 2}, {(0, 0): -0.1, (1, 0): 1.0, (2, 0): 0.0})
{(0, 0): 0.1, (0, 1): 0.1, (0, 2): 0.1, (1, 0): 0.0, (1, 1): 0.0, (1, 2): 0.0, (2, 0): 0.0, (2, 1): 0.0, (2, 2): 0.0, (3, 0): 0.0, (3, 1): 0.0, (3, 2): 0.0}
({0, 1, 2, 3}, {0, 1, 2}, {(0, 0): 1, (0, 1): 2, (0, 2): 3, (1, 0): 1, (1, 1): 1, (1, 2): 1, (2, 0): 2, (2, 1): 2, (2, 2): 2, (3, 0): 3, (3, 1): 3, (3, 2): 3}, {(0, 0): 0.1, (0, 1): 0.1, (0, 2): 0.1, (1, 0): 0.0, (1, 1): 0.0, (1, 2): 0.0, (2, 0): 0.0, (2, 1): 0.0, (2, 2): 0.0, (3, 0): 0.0, (3, 1): 0.0, (3, 2): 0.0})
{(0, 0): 0.1, (0, 1): 0.1, (1, 0): -0.1, (1, 1): 0.1, (2, 0): 0.1, (2, 1): -0.1, (3, 0): 0.0, (3, 1): 0.0}
({0, 1, 2, 3}, {0, 1}, {(0, 0): 1, (0, 1): 2, (1, 0): 1, (1, 1): 3, (2, 0): 3, (2, 1): 2, (3, 0): 3, (3, 1): 3}, {(0, 0): 0.1, (0, 1): 0.1, (1, 0): -0.1, (1, 1): 0.1, (2, 0): 0.1, (2, 1): -0.1, (3, 0): 0.0, (3, 1): 0.0})


## Interaction loop, trajectory, termination, truncation, episode

Общий цикл взаимодействия в рамках эпизода:
1. Инициализировать среду: $s \leftarrow \text{env.init()}$
2. Цикл:
    - выбрать действие: $a \leftarrow \pi(s)$
    - получить ответ от среды: $s, r, d \leftarrow \text{env.next(a)}$
    - если $d == \text{True}$, выйти из цикла

In [69]:
def run_episode(mdp):
    states, actions, T, R, termination = mdp
    agent = Agent(actions)

    s = 0
    tau = []
    for _ in range(5):
        a = agent.act(s)
        s_next = T[(s, a)]
        r = R[(s, a)]

        tau.append((s, a, r))
        s = s_next
        
        if s in termination:
            break
            
    return tau

term_a = [2]
term_b = [1,2,3]
term_c = [3]

A_mdp = *A_mdp, term_a
B_mdp = *B_mdp, term_b
C_mdp = *C_mdp, term_c

print(run_episode(A_mdp))
print(run_episode(A_mdp))
print(run_episode(B_mdp))
print(run_episode(B_mdp))
print(run_episode(C_mdp))
print(run_episode(C_mdp))

[(0, 0, -0.1), (1, 0, 1.0)]
[(0, 0, -0.1), (1, 0, 1.0)]
[(0, 1, 0.1)]
[(0, 0, 0.1)]
[(0, 1, 0.1), (2, 1, -0.1), (2, 0, 0.1)]
[(0, 1, 0.1), (2, 1, -0.1), (2, 1, -0.1), (2, 1, -0.1), (2, 0, 0.1)]


Termination — означает окончание эпизода, когда достигнуто терминальное состояние. Является частью задания среды.

Truncation — означает окончание эпизода, когда достигнут лимит по числу шагов (=времени). Обычно является внешне заданным параметром для удобства обучения.

Пока не будем вводить truncation, но поддержим termination: расширьте определение среды информацией о терминальных состояниях для всех описанных ранее сред. Сгенерируйте по несколько случайных траекторий для каждой среды.

### Return, expected return

Наиболее важная метрика оценки качества работы агента: отдача.

Отдача: $G(s_t) = \sum_{i=t}^T r_i$

Обычно также вводят параметр $\gamma \in [0, 1]$, дисконтирующий будущие вознаграждения. А еще, тк отдача может меняться от запуска к запуску благодаря вероятностным процессам, нас интересует отдача в среднем — ожидаемая отдача:

$$\hat{G}(s_t) = \mathbb{E} [ \sum_{i=t}^T \gamma^{i-t} r_i ]$$

Именно ее и оптимизируют в RL.

Давайте научимся считать отдачу для состояний по траектории и считать среднюю отдачу.

In [70]:
def get_disc_path(path, g):
    for t, step in enumerate(path):
        if t == 0:
            r_i = step[2]
            continue
            
        r = step[2]          
        r_i += (g ** t) * r 
        
        new_step = (step[0], step[1], r_i)
        path[t] = new_step
        
    return path

def get_mean_G(mdp, nums, g):
    states, actions, T, R, termination = mdp
    count = [0] * len(states)
    sum = [0] * len(states)
    ans = [0] * len(states)
    
    for _ in range(nums):
        episode = run_episode(mdp)   
        new_path =  get_disc_path(episode, g)
        for step in new_path:
            state, action, r = step
            count[state] += 1
            sum[state] += r
    
    for i in range(len(states)):
        if count[i] == 0:
            ans[i] = 0.0 
            continue
            
        ans[i] = sum[i] / count[i]
        
    return ans

print(get_mean_G(A_mdp, 100, 0.5))
print(get_mean_G(A_mdp, 1000, 0.5))
print(get_mean_G(B_mdp, 100, 0.5))
print(get_mean_G(B_mdp, 1000, 0.5))
print(get_mean_G(C_mdp, 100, 0.5))
print(get_mean_G(C_mdp, 1000, 0.5))

[-0.09999999999999981, 0.39999999999999925, 0.0]
[-0.09999999999999859, 0.39999999999999436, 0.0]
[0.09999999999999981, 0.0, 0.0, 0.0]
[0.09999999999999859, 0.0, 0.0, 0.0]
[0.09999999999999981, 0.07524271844660198, 0.08324275362318843, 0.0]
[0.09999999999999859, 0.07382726269315637, 0.07111378205128174, 0.0]
